# Concurrency, Parallelism & Asyncio: Beginner Guide

### 🌟 What is Concurrency, Multiprocessing & AsyncIO?
To maximize system performance, Python provides multiple execution models: **Multiprocessing** (for heavy CPU mathematical calculations using multiple CPU cores), **Threading** (for I/O operations), and **AsyncIO** (for high-concurrency event loops handling thousands of network requests).

This interactive guide loads and works directly with `data/raw_transactions.csv`, giving you real-world hands-on practice.

### 📚 Key Concepts Covered in this Notebook:
- **Multi-Threading**: Covers `threading.Thread` and `threading.Lock`.
- **Execution Pools**: Covers `concurrent.futures.ThreadPoolExecutor` and `ProcessPoolExecutor`.
- **Asynchronous Event Loop**: Covers `async def`, `await`, `asyncio.run()`, and `asyncio.gather()`.


In [1]:
# Setup imports & dataset loading from raw_transactions.csv
import csv
import sys
import time
import os
import functools
import contextlib
import asyncio
import threading
from dataclasses import dataclass
from typing import List, Dict, Optional, Union, Protocol, Literal, Final, TypedDict, Callable, TypeVar

csv_path = 'data/raw_transactions.csv' if os.path.exists('data/raw_transactions.csv') else '../data/raw_transactions.csv'
transactions = []
with open(csv_path, mode='r', encoding='utf-8') as f:
    reader = csv.DictReader(f)
    for row in reader:
        transactions.append(row)

print(f"Python Version: {sys.version.split()[0]}")
print(f"Loaded {len(transactions)} transaction records from {csv_path}")

Python Version: 3.12.7
Loaded 15000 transaction records from ../data/raw_transactions.csv


### 🔹 Multi-Threading: `threading.Thread`
Spawns OS threads sharing Python heap memory space (suitable for I/O-bound tasks). It provides a robust, standardized way to process, clean, and analyze datasets reliably and efficiently. **Tip:** Print intermediate variables with `print()` or inspect their type with `type()` to trace how data changes at each step.

**Syntax:** `t = threading.Thread(target=func, args=(...)); t.start(); t.join()`


In [2]:
def worker_audit(tx_id):
    print(f'Thread auditing {tx_id}')

t1 = threading.Thread(target=worker_audit, args=(transactions[0]['transaction_id'],))
t1.start()
t1.join()
print('Thread worker finished.')

Thread auditing TX110686
Thread worker finished.


### 🔹 Thread Synchronization: `threading.Lock`
Mutual exclusion lock preventing race conditions on shared memory across threads. Saving and loading data properly ensures results are persistent, shareable across teams, and ready for downstream analysis. **Tip:** Print intermediate variables with `print()` or inspect their type with `type()` to trace how data changes at each step.

**Syntax:** `with lock: shared_state += 1`


In [3]:
total_balance = 0.0
lock = threading.Lock()

def thread_safe_deposit(amt):
    global total_balance
    with lock:
        total_balance += amt

threads = [threading.Thread(target=thread_safe_deposit, args=(float(transactions[i]['transaction_amount']),)) for i in range(3)]
for t in threads: t.start()
for t in threads: t.join()
print(f'Thread-Safe Consolidated Balance: ${total_balance:,.2f}')

Thread-Safe Consolidated Balance: $1,677.98


### 🔹 Thread Pool Executor: `ThreadPoolExecutor`
High-level thread pool manager mapping worker functions across threads. It provides a robust, standardized way to process, clean, and analyze datasets reliably and efficiently. **Tip:** Print intermediate variables with `print()` or inspect their type with `type()` to trace how data changes at each step.

**Syntax:** `with ThreadPoolExecutor() as ex: results = ex.map(fn, items)`


In [4]:
from concurrent.futures import ThreadPoolExecutor
def process_item(t):
    return f"{t['transaction_id']}: Verified {t['card_type']}"

with ThreadPoolExecutor(max_workers=3) as pool:
    results = list(pool.map(process_item, transactions[:3]))
print('ThreadPoolExecutor results:', results)

ThreadPoolExecutor results: ['TX110686: Verified Visa', 'TX107170: Verified MasterCard', 'TX108328: Verified Discover']


### 🔹 Process Pool Executor: `ProcessPoolExecutor`
Spawns independent OS Python processes bypassing the GIL for CPU-bound computation. It provides a robust, standardized way to process, clean, and analyze datasets reliably and efficiently. **Tip:** Print intermediate variables with `print()` or inspect their type with `type()` to trace how data changes at each step.

**Syntax:** `with ProcessPoolExecutor() as ex: ...`


In [5]:
from concurrent.futures import ProcessPoolExecutor
print('ProcessPoolExecutor: Bypasses CPython GIL by spawning separate OS processes.')

ProcessPoolExecutor: Bypasses CPython GIL by spawning separate OS processes.


### 🔹 Asynchronous Coroutines: `async def` & `await`
Defines non-blocking coroutines yielding control back to event loop on async I/O. It provides a robust, standardized way to process, clean, and analyze datasets reliably and efficiently. **Tip:** Print intermediate variables with `print()` or inspect their type with `type()` to trace how data changes at each step.

**Syntax:** `async def fetch(): await asyncio.sleep(0.01)`


In [6]:
async def simulate_async_api(tx_id):
    await asyncio.sleep(0.01) # Non-blocking async sleep
    return f'{tx_id}: APPROVED'

print('Coroutine function defined.')

Coroutine function defined.


### 🔹 Event Loop Execution: `asyncio.run()`
Creates a new event loop, executes the main coroutine, and closes the loop. Saving and loading data properly ensures results are persistent, shareable across teams, and ready for downstream analysis. **Tip:** Always remember to include `self` as the first parameter in instance methods so Python knows which object instance is executing.

**Syntax:** `asyncio.run(main())`


In [7]:
import nest_asyncio
nest_asyncio.apply()
async def main():
    return await simulate_async_api(transactions[0]['transaction_id'])
print('asyncio.run result:', asyncio.run(main()))


asyncio.run result: TX110686: APPROVED


### 🔹 Concurrent Task Gathering: `asyncio.gather()`
Runs multiple async coroutines concurrently on single-threaded event loop. Saving and loading data properly ensures results are persistent, shareable across teams, and ready for downstream analysis. **Tip:** Print intermediate variables with `print()` or inspect their type with `type()` to trace how data changes at each step.

**Syntax:** `await asyncio.gather(*tasks)`


In [8]:
import nest_asyncio
nest_asyncio.apply()
async def run_batch():
    tasks = [simulate_async_api(t['transaction_id']) for t in transactions[:3]]
    return await asyncio.gather(*tasks)
print('asyncio.gather results:', asyncio.run(run_batch()))


asyncio.gather results: ['TX110686: APPROVED', 'TX107170: APPROVED', 'TX108328: APPROVED']


## 💡 Real-World Practice & Scenarios
Practical scenarios and common data questions explained simply with real examples.


### 🔍 Scenario: Q1: When to choose Threading vs Multiprocessing vs Asyncio

**Approach:** Explain trade-offs: (1) Threading for I/O blocking calls; (2) Multiprocessing for CPU bound GIL bypass; (3) Asyncio for high concurrency non-blocking network sockets.
**Syntax:** `ProcessPoolExecutor` vs `ThreadPoolExecutor` vs `asyncio`


In [9]:
print('CPU-Bound: Multiprocessing (separate OS processes).')
print('I/O-Bound High Concurrency: Asyncio (event loop coroutines).')
print('I/O-Bound Blocking APIs: Threading.')

CPU-Bound: Multiprocessing (separate OS processes).
I/O-Bound High Concurrency: Asyncio (event loop coroutines).
I/O-Bound Blocking APIs: Threading.
